# Unified Message Passing (UniMP) on OGBN-Arxiv

Node Classification on ogbn-arxiv: Combining masked node label propagation with feature-based graph transformers. This notebook implements the approach with `TransformerConv` inside a `K3UniMP` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `TransformerConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node import models as k3_models
from k3_node.datasets import Planetoid
from k3_node import transforms as k3_transforms

title = "Unified Message Passing (UniMP) with MaskLabel & TransformerConv"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora", transform=k3_transforms.NormalizeFeatures())
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. UniMP Model with MaskLabel & TransformerConv
class K3UniMP(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels, num_classes):
        super().__init__()
        # method="concat" appends the label embedding to x, so out_channels
        # here is the label-embedding width (num_classes), matching
        # in_channels = num_features + num_classes below.
        self.mask_label = k3_models.MaskLabel(num_classes=num_classes, out_channels=num_classes, method="concat")
        self.lin_in = layers.Dense(hidden_channels)
        self.conv1 = k3_layers.TransformerConv(hidden_channels, hidden_channels, heads=2)
        self.conv2 = k3_layers.TransformerConv(hidden_channels * 2, out_channels, heads=1, concat=False)

    def call(self, x, edge_index, y=None, mask=None):
        if y is not None and mask is not None:
            x = self.mask_label(x, y, mask)
        h = ops.relu(self.lin_in(x))
        h = ops.relu(self.conv1(h, edge_index))
        return self.conv2(h, edge_index)

k3_model = K3UniMP(num_features + num_classes, 64, num_classes, num_classes)

# 3. Forward Pass Test
out = k3_model(data.x, data.edge_index, data.y, data.train_mask)
print(f"UniMP forward pass completed! Output shape: {out.shape}")

print("\n✓ K3-Node UniMP execution completed successfully!")